In [1]:
import json
from sentence_transformers import SentenceTransformer
import faiss
from openai import OpenAI
import os
from dotenv import load_dotenv
from serpapi.google_search import GoogleSearch

c:\Users\Acer\Documents\RAG QA\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def preprocess_data(movies):
    processed_chunks = []
    
    for movie in movies:
        movie_text = f"Title: {movie['title']}. Year: {movie['year']}. Plot: {movie['plot']}. Actors: {movie['actors']}. Director: {movie['director']}."
        processed_chunks.append(movie_text)   
    return processed_chunks

In [3]:
input_data_path = "..\data\movie_trivia.json"
with open(input_data_path, 'r') as f:
    movies = json.load(f)

In [4]:
processed_data = preprocess_data(movies)

In [5]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [6]:
def create_vector_db(processed_data):
    embeddings = embedding_model.encode(processed_data)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    return index

In [7]:
index = create_vector_db(processed_data)

In [8]:
def retrieve(query, index, chunks, top_k=3):
    query_embedding = embedding_model.encode([query])
    scores, indices = index.search(query_embedding, top_k)
    retrieved_chunks = []
    for idx in indices[0]:
        if idx != -1: 
            retrieved_chunks.append(chunks[idx])
    return retrieved_chunks

In [9]:
load_dotenv()
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [49]:
def generate_answer(question, context):
    if not context:
        return None
    
    prompt =f"""
        You are a movie trivia assistant.

        Use only the following context to answer the question. If the context does not contain the answer, reply with "I don't know."

        Context:
        {context}

        Question:
        {question}

        Answer:
        """.strip()

    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

In [ ]:
def search_web(query):
    try:
        params = {
            "q": query,
            "hl": "en",
            "gl": "us",
            "api_key": os.getenv("SERPAPI_KEY")
        }
        search = GoogleSearch(params)
        results = search.get_dict()
        description = results.get("knowledge_graph", {}).get("description", "")
        snippets = [result.get("snippet", "") for result in results.get("organic_results", []) if result.get("snippet")]
        combined_context = description + " " + " ".join(snippets[:3]) 

        return combined_context.strip()
    except:
        return None

In [59]:
def ask_question(question):
    retrieved_chunks = retrieve(question, index, processed_data)  
    context = "\n---\n".join(retrieved_chunks)
    print("Question")
    if not retrieved_chunks:
        print("I couldn’t find information about this.")
    else:
        answer = generate_answer(question, context)
        if "i don't know" in answer.lower():
            print("Searching Web...")
            results = search_web(question)
            if results:
                answer = generate_answer(question, results)
                if "i don't know" not in answer.lower():
                    print(f"Answer: {answer}")
                    return
        else:
                print(f"Answer: {answer}")
                return
    print("I couldn't find an answer.  Please try rephrasing your question.")


In [61]:
ask_question("Who directed The Shawshank Redemption?")
ask_question("Latest Oscar winners") 
ask_question("Who directed captain marvel?")

Question
Answer: Frank Darabont.
Question
Searching Web...
Answer: The latest Oscar winners from the 97th Academy Awards include "Anora" for Best Picture, Sean Baker for Best Director and Best Original Screenplay, Mikey Madison for Best Actress in "Anora," Adrien Brody for Best Actor in "The Brutalist," Zoe Saldaña for Best Supporting Actress in "Emilia Pérez," and Kieran Culkin for Best Supporting Actor in "A Real Pain."
Question
Searching Web...
Answer: Anna Boden and Ryan Fleck directed Captain Marvel.
